# Regelleistung.net
Required data:
 - Activation Price (CBMP)
 - Capacity Price

In [6]:
import pandas as pd
import requests
import io
import warnings
import os

# Unterdrückt die irrelevante Openpyxl-Warnung wegen fehlender Styles
warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')

def fetch_and_parse_regelleistung(year, market_type):
    """
    Lädt aFRR Jahres-Daten herunter und extrahiert Preise sowie Volumina.
    """
    url = f"https://www.regelleistung.net/apps/cpp-publisher/api/v2/tenders/files/RESULT_OVERVIEW_{market_type}_MARKET_aFRR_{year}-01-01_{year}-12-31.xlsx"
    
    print(f"Lade {market_type}-Daten für {year} herunter... (Das kann kurz dauern)")
    try:
        # User-Agent mitsenden, damit die Webseite den Request nicht blockiert
        response = requests.get(url, timeout=60, headers={'User-Agent': 'Mozilla/5.0'})
        response.raise_for_status()
    except Exception as e:
        print(f"  -> Überspringe {year} {market_type}: Datei nicht gefunden (ggf. noch keine Daten für das Jahr).")
        return pd.DataFrame()

    # Excel direkt aus dem Arbeitsspeicher (RAM) in Pandas laden
    df = pd.read_excel(io.BytesIO(response.content), engine='openpyxl')
    df.columns = df.columns.str.strip() 

    # --- 1. DYNAMISCHE SPALTEN-SUCHE ---
    date_col = [c for c in df.columns if 'DATE' in c.upper() or 'DATUM' in c.upper()][0]
    prod_col = [c for c in df.columns if 'PRODUC' in c.upper() or 'PRODUKT' in c.upper() or 'TIME' in c.upper()][0]

    # --- 2. ZEITSTEMPEL PARSEN ---
    if df[prod_col].astype(str).str.contains('_').any():
        df['quarter_hour_int'] = df[prod_col].str.extract(r'_(\d+)').astype(int)
        df['time_offset'] = pd.to_timedelta((df['quarter_hour_int'] - 1) * 15, unit='m')
        df['timestamp'] = pd.to_datetime(df[date_col]) + df['time_offset']
        df['Richtung'] = df[prod_col].str.split('_').str[0]
    else: # 2022 Capacity Format
        df['start_time'] = df[prod_col].astype(str).str.split(' - ').str[0]
        df['timestamp'] = pd.to_datetime(df[date_col].astype(str) + ' ' + df['start_time'])
        dir_col = [c for c in df.columns if 'DIRECTION' in c.upper() or 'RICHTUNG' in c.upper()][0]
        df['Richtung'] = df[dir_col]

    df['timestamp'] = df['timestamp'].dt.tz_localize('Europe/Berlin', ambiguous='NaT', nonexistent='NaT').dt.tz_convert('UTC')
    df.dropna(subset=['timestamp'], inplace=True)

    # --- 3. RELEVANTE FEATURE-SPALTEN FINDEN ---
    # Dictionary mit "Original-Name": "Neuer-Modell-Name"
    val_cols = {}
    if market_type == 'ENERGY':
        marg_cols = [c for c in df.columns if 'MARGINAL_ENERGY_PRICE' in c.upper()]
        avg_cols = [c for c in df.columns if 'AVERAGE_ENERGY_PRICE' in c.upper()]
        off_cols = [c for c in df.columns if 'OFFERED_CAPACITY' in c.upper()]
        
        if marg_cols: val_cols[marg_cols[0]] = 'afrr_activation_price'
        if avg_cols: val_cols[avg_cols[0]] = 'afrr_activation_avg_price'
        if off_cols: val_cols[off_cols[0]] = 'afrr_activation_offered_mw'
    else:
        marg_cols = [c for c in df.columns if 'MARGINAL_CAPACITY_PRICE' in c.upper() or 'GRENZWERT' in c.upper()]
        off_cols = [c for c in df.columns if 'OFFERED_CAPACITY' in c.upper()]
        
        if marg_cols: val_cols[marg_cols[0]] = 'afrr_capacity_price'
        if off_cols: val_cols[off_cols[0]] = 'afrr_capacity_offered_mw'

    for col in val_cols.keys():
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0)

    # --- 4. PIVOTISIEREN (Multi-Spalten) UND RESAMPLEN ---
    df_clean = df[['timestamp', 'Richtung'] + list(val_cols.keys())]
    df_pivot = df_clean.pivot_table(index='timestamp', columns='Richtung', values=list(val_cols.keys()))

    # Spaltennamen flachmachen (z.B. aus 'MARGINAL_PRICE', 'NEG' wird 'afrr_activation_price_neg')
    new_cols = []
    for col in df_pivot.columns:
        orig_metric_name = col[0]
        direction = col[1].replace('ATIVE', '').lower() # POSITIVE -> pos, NEGATIVE -> neg
        new_base_name = val_cols[orig_metric_name]
        new_cols.append(f"{new_base_name}_{direction}")
    df_pivot.columns = new_cols

    df_pivot = df_pivot.resample('15min').ffill()
    return df_pivot


# ==============================================================================
# HAUPT-SCHLEIFE (2022 bis 2025)
# ==============================================================================
years = [2022, 2023, 2024, 2025]
all_years_data = []

for y in years:
    df_cap = fetch_and_parse_regelleistung(y, 'CAPACITY')
    df_ene = fetch_and_parse_regelleistung(y, 'ENERGY')
    
    # Jahr zusammenfügen, falls Daten existieren
    if not df_cap.empty or not df_ene.empty:
        df_year = pd.concat([df_cap, df_ene], axis=1)
        all_years_data.append(df_year)

# 1. Alle Jahre kombinieren
df_master = pd.concat(all_years_data)

# 2. Sortieren und Duplikate (z.B. durch Zeitumstellung) entfernen
df_master = df_master.sort_index()
df_master = df_master[~df_master.index.duplicated(keep='first')]

# 3. Speichern
output_dir = "../data"
os.makedirs(output_dir, exist_ok=True)
output_path = f"{output_dir}/regelleistung.parquet"

df_master.to_parquet(output_path)

print(f"\n--- ERFOLG! ---")
print(f"Parquet-Datei mit {len(df_master)} Zeilen erfolgreich unter '{output_path}' gespeichert.")
print(f"Zeitraum: {df_master.index.min()} bis {df_master.index.max()}")

Lade CAPACITY-Daten für 2022 herunter... (Das kann kurz dauern)
Lade ENERGY-Daten für 2022 herunter... (Das kann kurz dauern)
Lade CAPACITY-Daten für 2023 herunter... (Das kann kurz dauern)
Lade ENERGY-Daten für 2023 herunter... (Das kann kurz dauern)
Lade CAPACITY-Daten für 2024 herunter... (Das kann kurz dauern)
Lade ENERGY-Daten für 2024 herunter... (Das kann kurz dauern)
Lade CAPACITY-Daten für 2025 herunter... (Das kann kurz dauern)
Lade ENERGY-Daten für 2025 herunter... (Das kann kurz dauern)

--- ERFOLG! ---
Parquet-Datei mit 140257 Zeilen erfolgreich unter '../data/regelleistung.parquet' gespeichert.
Zeitraum: 2021-12-31 22:45:00+00:00 bis 2025-12-31 22:45:00+00:00
